In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
df = pd.read_csv("shop_smart_ecommerce.csv")
df

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.200000,0.200000,0.000000,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.000000,0.100000,0.000000,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.200000,0.200000,0.000000,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.050000,0.140000,0.000000,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.020000,0.050000,0.000000,0.0,Feb,3,3,1,4,Returning_Visitor,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12325,3,145.0,0,0.0,53,1783.791667,0.007143,0.029031,12.241717,0.0,Dec,4,6,1,1,Returning_Visitor,True,False
12326,0,0.0,0,0.0,5,465.750000,0.000000,0.021333,0.000000,0.0,Nov,3,2,1,8,Returning_Visitor,True,False
12327,0,0.0,0,0.0,6,184.250000,0.083333,0.086667,0.000000,0.0,Nov,3,2,1,13,Returning_Visitor,True,False
12328,4,75.0,0,0.0,15,346.000000,0.000000,0.021053,0.000000,0.0,Nov,2,2,3,11,Returning_Visitor,False,False


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

In [4]:
df["Weekend"] = df["Weekend"].astype(int)
x = df.drop("Revenue",axis=1).copy()
y = df["Revenue"].astype(int)
xtr,xts,ytr,yts = train_test_split(x,y,test_size=0.20,random_state = 42)

In [5]:
num_features = x.select_dtypes(include="number").columns
cat_features = x.select_dtypes(include=["object"]).columns

preprocessor = ColumnTransformer([
    ('num',StandardScaler(),num_features),
    ('cat',OneHotEncoder(handle_unknown="ignore"),cat_features)
])

pip = Pipeline([
    ('preprocess',preprocessor),
    ('model',DecisionTreeClassifier())
])

In [6]:
xtr_scaled = pip.named_steps['preprocess'].fit_transform(xtr, ytr)
model = DecisionTreeClassifier()
path = model.cost_complexity_pruning_path(xtr_scaled, ytr)
ccp_alphas = path.ccp_alphas

In [7]:
# DecisionTreeClassifier().get_params()

param_grid = {
    'model__max_depth' : [5,8,10,15,18,20],
    'model__ccp_alpha' : ccp_alphas,
    "model__min_samples_leaf": [20, 30, 50]
}

cv = GridSearchCV(
    pip,
    param_grid,
    cv = 5,
    scoring="f1"
)

In [8]:
cv.fit(xtr,ytr)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocess',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         Index(['Administrative', 'Administrative_Duration', 'Informational',
       'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
       'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay',
       'OperatingSystems', 'Browser', 'Region', 'TrafficType...
       4.26394258e-04, 4.27480400e-04, 4.85026496e-04, 4.85789923e-04,
       5.06250080e-04, 5.61332385e-04, 6.14689720e-04, 6.40149330e-04,
       7.42489055e-04, 8.32401635e-04, 1.10975936e-03, 1.34332459e-03,
       1.90601342e-03, 2.32720179e-03, 2.55182630e-03, 3.41064638e-03,
       3.65950993e-03, 1.01473548e-02, 9.36090799e-02]),
                         'model__max_depth': [5, 8, 10, 15, 18, 20],
                         'model__min_samples_leaf': [20, 30, 50]},
             scoring='f1')

In [9]:
print(f"Best F1 score : {cv.best_score_}")
print(f"Best params : {cv.best_params_}")

Best F1 score : 0.6508526754654682
Best params : {'model__ccp_alpha': np.float64(0.0), 'model__max_depth': 8, 'model__min_samples_leaf': 50}
